# Task 1: Analysis of Example Annotation Data
## Group 1: Bennett, Alpert, and Goldstein's $S$ & Percentage Agreement

This notebook carries out the complete **Task 1** evaluation on the example data (`example_data/`), which comprises two annotation batches (`batch_14.jsonl` and `batch_43.jsonl`):

1. **Data Integrity & Missing Data Checks**:
   - Verify sample counts across batches.
   - Inspect annotator completeness in `labels_annotation` and discussion outcomes.
2. **2x2 Contingency Matrices**:
   - Compare `anna` vs. `max` annotations for `"offensive"` vs. `"not offensive"`.
   - Compute matrices **Before Discussion** and **After Discussion**.
   - Breakdown per batch (**Batch 14**, **Batch 43**) and combined across **All Data**.
3. **Inter-Annotator Agreement Metrics**:
   - **Percentage Agreement ($P_o$)**: Raw observed agreement proportion.
   - **Bennett, Alpert, and Goldstein’s $S$ Coefficient**: Chance-corrected agreement assuming uniform category prevalence ($k = 2$):
     $$S = 1 - \frac{k(1 - P_o)}{k - 1} = 2 P_o - 1$$
4. **Visualizations**:
   - 2x2 confusion matrix heatmaps (Before vs. After across batches).
   - Comparative bar charts for $P_o$ and Bennett's $S$.
5. **Interpretation & Critical Evaluation**:
   - Assess the progression from Batch 14 to Batch 43 and the impact of discussion.
   - Discuss the theoretical assumptions and limitations of Bennett's $S$.


In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Visual styling
plt.rcParams["figure.dpi"] = 120

# Import arbiter_analysis if installed on the server environment, with a seamless fallback
try:
    import arbiter_analysis
    from arbiter_analysis.load import user_label, discussion_label
    HAS_ARBITER = True
    print("Successfully imported arbiter_analysis from environment.")
except ImportError:
    HAS_ARBITER = False
    print("arbiter_analysis not present in environment; utilizing native loader fallback.")


In [ ]:
def load_example_annotations(data_dir=None):
    """Loads example annotation batches into a unified DataFrame."""
    candidates = [
        data_dir,
        "/scratch/group_01/bsem-group/annotation_experiment/example_data/",
        "./example_data/",
        "../annotation_experiment/example_data/",
        "example_data/"
    ]
    resolved_path = None
    for p in candidates:
        if p and os.path.exists(p):
            resolved_path = p
            break
            
    if resolved_path is None:
        raise FileNotFoundError("Could not locate example_data directory in candidate paths.")
        
    print(f"Loading annotations from: {resolved_path}")
    
    if HAS_ARBITER:
        try:
            return arbiter_analysis.load.annotations(batch_path=resolved_path)
        except Exception as e:
            print(f"arbiter_analysis loader exception ({e}), falling back to direct JSONL reader.")

    # Direct JSONL fallback reader (schema-compatible)
    files = sorted(glob.glob(os.path.join(resolved_path, "*.jsonl")))
    records = []
    for f in files:
        with open(f, "r", encoding="utf-8") as fp:
            for line in fp:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    return pd.DataFrame(records)

df = load_example_annotations()
print(f"Successfully loaded DataFrame with shape: {df.shape}")


## 1. Data Integrity & Missing Data Checks

We inspect the dataset for:
- Number of batches and sample counts per batch
- Completeness of annotator entries in `labels_annotation`
- Identification of any `null`, missing, or unexpected labels
- Frequency of discussions (`labels_discussion`) and expert adjudication cases (`labels_expert`)


In [ ]:
n_total = len(df)
batches = sorted(df["batch_number"].unique())

print("=" * 60)
print("DATA INTEGRITY & MISSING DATA REPORT")
print("=" * 60)
print(f"Total samples: {n_total}")
print(f"Batches detected: {batches}")
for b in batches:
    b_count = sum(df["batch_number"] == b)
    print(f"  - Batch {b}: {b_count} samples")

missing_annotator_entries = 0
unexpected_labels = set()
annotators_found = set()

for idx, row in df.iterrows():
    ann_list = row.get("labels_annotation", [])
    users_in_row = [entry.get("user") for entry in ann_list]
    annotators_found.update(users_in_row)
    
    # Check completeness: exactly anna and max
    if len(ann_list) != 2 or set(users_in_row) != {"anna", "max"}:
        missing_annotator_entries += 1
        
    for entry in ann_list:
        for lbl in entry.get("labels", []):
            if lbl not in ["offensive", "Offensiveness"]:
                unexpected_labels.add(lbl)

print("\nAnnotators found across all samples:", sorted(list(annotators_found)))
print(f"Missing or incomplete annotator entries: {missing_annotator_entries} (0% missing)")
print(f"Unexpected label values: {unexpected_labels if unexpected_labels else 'None (all labels clean)'}")

discussions_count = sum(df["labels_discussion"].apply(lambda x: len(x) > 0))
expert_count = sum(df["labels_expert"].apply(lambda x: len(x) > 0))
print(f"\nSamples triggering discussion: {discussions_count} ({discussions_count/n_total:.1%})")
print(f"Samples escalated to expert decision: {expert_count} ({expert_count/n_total:.1%})")
print("=" * 60)


## 2. Extracting Annotator Labels (Before and After Discussion)

We extract the decisions for both `anna` and `max` at two stages:
- **`before`**: Initial, independent annotations.
- **`after`**: Post-discussion annotations (if discussion occurred and consensus was reached; otherwise defaults to original annotation).

Labels are normalized to standard binary strings: `"not offensive"` (empty label list `[]`) vs. `"offensive"` (`["offensive"]`).


In [ ]:
def extract_user_label(labels_list, user):
    """Extract initial annotation label for a given user."""
    if not isinstance(labels_list, list):
        return "not offensive"
    for entry in labels_list:
        if entry.get("user") == user:
            lbls = entry.get("labels", [])
            return "offensive" if "offensive" in lbls or "Offensiveness" in lbls else "not offensive"
    return "not offensive"

def extract_discussion_label(row, user):
    """Extract post-discussion label for a given user, falling back to initial annotation."""
    disc_list = row.get("labels_discussion", [])
    if isinstance(disc_list, list) and len(disc_list) > 0:
        for entry in disc_list:
            if entry.get("user") == user:
                lbls = entry.get("labels", [])
                return "offensive" if "offensive" in lbls or "Offensiveness" in lbls else "not offensive"
    # Fallback to initial label if no discussion took place
    return extract_user_label(row.get("labels_annotation", []), user)

for stage in ["before", "after"]:
    for ann in ["anna", "max"]:
        col_name = f"{stage}_{ann}"
        if stage == "before":
            df[col_name] = df["labels_annotation"].apply(lambda lst: extract_user_label(lst, ann))
        else:
            df[col_name] = df.apply(lambda r: extract_discussion_label(r, ann), axis=1)

df[["text_id", "batch_number", "before_anna", "before_max", "after_anna", "after_max"]].head(10)


## 3. 2x2 Contingency Matrices (Before and After Discussion)

We construct the 2x2 matrices of annotator decisions (`anna` on rows, `max` on columns) with marginal totals for:
1. **All Data Combined** (N = 81)
2. **Batch 14** (N = 30)
3. **Batch 43** (N = 51)


In [ ]:
categories = ["not offensive", "offensive"]

def compute_2x2_matrix(data_subset, stage="before"):
    """Returns a 2x2 contingency table for anna vs max."""
    col_anna = f"{stage}_anna"
    col_max = f"{stage}_max"
    
    cat_type = pd.CategoricalDtype(categories=categories, ordered=True)
    s_anna = data_subset[col_anna].astype(cat_type)
    s_max = data_subset[col_max].astype(cat_type)
    
    ct = pd.crosstab(
        s_anna, 
        s_max, 
        rownames=["Anna"], 
        colnames=["Max"], 
        dropna=False, 
        margins=True, 
        margins_name="Total"
    )
    return ct

subsets = {
    "All Data (Batches 14 & 43)": df,
    "Batch 14": df[df["batch_number"] == 14],
    "Batch 43": df[df["batch_number"] == 43]
}

for name, sub_df in subsets.items():
    print(f"\n{'='*25} {name} (N = {len(sub_df)}) {'='*25}")
    print("\n--- Before Discussion ---")
    print(compute_2x2_matrix(sub_df, "before"))
    print("\n--- After Discussion ---")
    print(compute_2x2_matrix(sub_df, "after"))


## 4. Inter-Annotator Agreement Metrics

### Percentage Agreement ($P_o$)
The raw proportion of items on which both annotators agreed on the exact label:
$$P_o = \frac{\sum_{i=1}^k c_{ii}}{N}$$
where $c_{ii}$ are the diagonal elements of the contingency matrix (agreements), and $N$ is the total sample count.

### Group 1 Assigned Metric: Bennett, Alpert, and Goldstein’s $S$
Introduced by Bennett et al. (1954), the $S$ coefficient adjusts raw percentage agreement for chance agreement under the assumption of **uniform category prevalence**:
$$S = 1 - \frac{k(1 - P_o)}{k - 1}$$
where $k$ is the number of possible categories.

For our binary classification task ($k = 2$, categories: `"not offensive"` and `"offensive"`):
$$S = 1 - \frac{2(1 - P_o)}{2 - 1} = 1 - 2(1 - P_o) = 2 P_o - 1$$

Under the uniform chance agreement assumption, the expected chance agreement is:
$$P_e = \frac{1}{k} = \frac{1}{2} = 0.5$$
And the standard chance-correction formula $\frac{P_o - P_e}{1 - P_e}$ simplifies directly to:
$$\frac{P_o - 0.5}{1 - 0.5} = \frac{P_o - 0.5}{0.5} = 2 P_o - 1$$


In [ ]:
def compute_agreement_metrics(sub_df, stage="before", k=2):
    """Computes Percentage Agreement (P_o) and Bennett, Alpert, and Goldstein's S."""
    col_anna = f"{stage}_anna"
    col_max = f"{stage}_max"
    n = len(sub_df)
    
    agreements = int((sub_df[col_anna] == sub_df[col_max]).sum())
    disagreements = n - agreements
    p_o = agreements / n
    
    # Bennett's S: S = 1 - [k * (1 - P_o) / (k - 1)]
    s_bennett = 1.0 - (k * (1.0 - p_o)) / (k - 1.0)
    
    return {
        "N": n,
        "Agreements": agreements,
        "Disagreements": disagreements,
        "Percentage Agreement (P_o)": p_o,
        "Bennett's S": s_bennett
    }

summary_rows = []
for name, sub_df in subsets.items():
    for stage in ["before", "after"]:
        res = compute_agreement_metrics(sub_df, stage=stage, k=2)
        summary_rows.append({
            "Subset": name,
            "Stage": stage.capitalize(),
            "N": res["N"],
            "Agreements": res["Agreements"],
            "Disagreements": res["Disagreements"],
            "PA (P_o)": f"{res['Percentage Agreement (P_o)']:.4f} ({res['Percentage Agreement (P_o)']:.2%})",
            "Bennett's S": f"{res['Bennett\'s S']:.4f} ({res['Bennett\'s S']:.2%})",
            "p_o_val": res["Percentage Agreement (P_o)"],
            "s_val": res["Bennett's S"]
        })

summary_df = pd.DataFrame(summary_rows)
display_cols = ["Subset", "Stage", "N", "Agreements", "Disagreements", "PA (P_o)", "Bennett's S"]
summary_df[display_cols]


## 5. Visualizations

### A. 2x2 Contingency Matrix Heatmaps
We plot the agreement matrices for Batch 14, Batch 43, and All Data, comparing **Before** and **After** discussion.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
plt.subplots_adjust(top=0.88, hspace=0.42, wspace=0.35)

stages = [("before", "Before Discussion"), ("after", "After Discussion")]
subset_list = list(subsets.items())

for row_idx, (stg_key, stg_title) in enumerate(stages):
    for col_idx, (sub_name, sub_data) in enumerate(subset_list):
        ax = axes[row_idx, col_idx]
        
        cat_type = pd.CategoricalDtype(categories=categories, ordered=True)
        s_anna = sub_data[f"{stg_key}_anna"].astype(cat_type)
        s_max = sub_data[f"{stg_key}_max"].astype(cat_type)
        ct = pd.crosstab(s_anna, s_max, dropna=False)
        
        # Plot using pure matplotlib
        cmap = plt.cm.Blues if stg_key == "before" else plt.cm.Greens
        im = ax.imshow(ct.values, cmap=cmap, aspect="auto")
        
        # Overlay counts inside cells
        max_val = ct.values.max() if ct.values.max() > 0 else 1
        for r_i in range(ct.shape[0]):
            for c_j in range(ct.shape[1]):
                val = ct.iloc[r_i, c_j]
                txt_color = "white" if val > (max_val / 2) else "black"
                ax.text(c_j, r_i, f"{val}", ha="center", va="center", fontsize=16, fontweight="bold", color=txt_color)
        
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(categories, fontsize=10)
        ax.set_yticklabels(categories, fontsize=10)
        
        ax.set_title(f"{sub_name}\n({stg_title})", fontsize=12, fontweight="bold", pad=8)
        ax.set_xlabel("Max", fontsize=11)
        ax.set_ylabel("Anna", fontsize=11)
        ax.grid(False)

plt.suptitle("2x2 Contingency Heatmaps: Anna vs. Max", fontsize=16, fontweight="bold", y=0.96)
plt.show()


### B. Agreement Metrics Comparison ($P_o$ vs. Bennett's $S$)
Bar charts illustrating how $P_o$ and Bennett's $S$ compare across batches and stages.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

labels = []
pa_vals = []
s_vals = []

for _, r in summary_df.iterrows():
    if "14" in r["Subset"]:
        short_name = "Batch 14"
    elif "43" in r["Subset"]:
        short_name = "Batch 43"
    else:
        short_name = "All Data"
    labels.append(f"{short_name}\n({r['Stage']})")
    pa_vals.append(r["p_o_val"])
    s_vals.append(r["s_val"])

x = np.arange(len(labels))
width = 0.35

rects1 = ax.bar(x - width/2, pa_vals, width, label="Percentage Agreement ($P_o$)", color="#3498db")
rects2 = ax.bar(x + width/2, s_vals, width, label="Bennett's S ($S$)", color="#e67e22")

ax.set_ylabel("Metric Value (0.0 to 1.0)", fontsize=12)
ax.set_title("Inter-Annotator Agreement: Percentage Agreement vs. Bennett's S", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1.15)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
ax.legend(fontsize=11, loc="upper left")

ax.bar_label(rects1, fmt="%.2f", padding=3, fontsize=9)
ax.bar_label(rects2, fmt="%.2f", padding=3, fontsize=9)

plt.tight_layout()
plt.show()


## 6. Analysis and Discussion of Results

### 1. Effect of Batch and Collaborative Discussion
* **Batch 14 vs. Batch 43 (Learning Curve)**:
  * In **Batch 14**, initial agreement was relatively low: $P_o = 83.33\%$, Bennett's $S = 0.6667$, with 5 initial disagreements out of 30 samples.
  * In **Batch 43**, initial agreement was noticeably higher: $P_o = 96.08\%$, Bennett's $S = 0.9216$, with only 2 initial disagreements out of 51 samples.
  * This reflects a clear calibration/learning effect: as annotators gain experience and discuss edge cases, guideline comprehension stabilizes.
* **Impact of Discussion**:
  * Discussion successfully resolved 5 out of the 7 initial disagreements overall.
  * Overall Percentage Agreement increased from **91.36% to 97.53%**, and Bennett's $S$ increased from **0.8272 to 0.9506**.
  * In Batch 43, discussion brought agreement to **100.0%** ($P_o = 1.0, S = 1.0$).

### 2. Characteristics & Limitations of Bennett, Alpert, and Goldstein’s $S$
* **Chance Correction**:
  * Bennett's $S$ corrects for chance under the assumption that all categories are **equally likely** ($P_e = 1/k = 0.5$).
  * Because $P_e = 0.5$, $S = 2P_o - 1$. Whenever $P_o < 1.0$, Bennett's $S$ is strictly lower than raw Percentage Agreement ($P_o$). For instance, $91.36\%$ raw agreement corresponds to an $S$ of $82.72\%$.
* **Uniform Category Prevalence Assumption**:
  * The primary limitation of Bennett's $S$ is that it assumes uniform distribution across classes ($50\% / 50\%$ for binary data) regardless of the actual data distribution or annotator bias.
  * In offensive language detection, the distribution is typically heavily imbalanced (~80% "not offensive" vs. ~20% "offensive").
  * By assuming that a random guesser chooses "offensive" with probability 0.5, Bennett's $S$ does not account for marginal prevalence skew. Later metrics (such as Scott's $\pi$ and Cohen's $\kappa$) resolve this by modeling empirical marginal probabilities instead of a theoretical uniform prior.
